# SqueezeNet — AlexNet-level accuracy with 50x fewer parameters (2016)

_Papers · Efficiency & Compression_

**Squeeze channels down with a 1×1 conv, then expand back out with a cheap mix of 1×1 and 3×3 convs — a tiny net that matches a big one.**

---

This notebook is a practice scaffold for the **SqueezeNet — AlexNet-level accuracy with 50x fewer parameters (2016)** lesson. We build it up one small step at a time: read the note above each cell, run the cell, watch what changes, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## Reference implementation — PyTorch

We build SqueezeNet's idea up in four steps: (1) count one Fire module's parameters by hand and confirm against PyTorch, (2) define the Fire module itself, (3) wire a tiny Fire-based net and a plain-conv baseline, then (4) train both on a toy task and sweep the squeeze ratio to see the params-vs-accuracy trade-off.

### Step 1 — Count a Fire module's parameters by hand

A convolution has `in × out × k × k` weights plus `out` biases. A **Fire module** is three convolutions: a `1×1` *squeeze* that thins the channels, then two parallel *expand* convs (`1×1` and `3×3`) that widen them back out.

We tally those three pieces for one concrete Fire module, then build the same (BatchNorm-free) layers in PyTorch and check that its own parameter count matches our arithmetic exactly.

In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(0)

# Fire(C=32, s1x1=8, e1x1=32, e3x3=32). A conv has in*out*k*k weights + out biases.
C, s1, e1, e3 = 32, 8, 32, 32
squeeze = C * s1 * 1 * 1 + s1        # 1x1 squeeze: 32->8
exp1 = s1 * e1 * 1 * 1 + e1          # 1x1 expand: 8->32
exp3 = s1 * e3 * 3 * 3 + e3          # 3x3 expand: 8->32
total = squeeze + exp1 + exp3
print("squeeze =", squeeze, " expand1x1 =", exp1, " expand3x3 =", exp3, " Fire total =", total)
# squeeze = 264  expand1x1 = 288  expand3x3 = 2336  Fire total = 2888

# Verify against PyTorch's own count for the SAME (no-BatchNorm) module.
class FireParamOracle(nn.Module):
    def __init__(self, cin, s1, e1, e3):
        super().__init__()
        self.sq = nn.Conv2d(cin, s1, 1)
        self.e1 = nn.Conv2d(s1, e1, 1)
        self.e3 = nn.Conv2d(s1, e3, 3, padding=1)

pt = sum(p.numel() for p in FireParamOracle(32, 8, 32, 32).parameters())
print("PyTorch Fire params =", pt, " match:", pt == total)
# PyTorch Fire params = 2888  match: True

### Step 2 — Define the Fire module

Now we build the real Fire module used for training. The `1×1` squeeze reduces the channel count; ReLU follows. Two parallel expands (`1×1` and `3×3`, the `3×3` padded so spatial size is preserved) then widen it back, and their outputs are concatenated along the channel axis.

We add BatchNorm after the squeeze and after the concatenated expand so the thin squeeze layers train stably.

In [ ]:
# The Fire module. BatchNorm added so thin squeezes train.
class Fire(nn.Module):
    def __init__(self, in_ch, s1x1, e1x1, e3x3):
        super().__init__()
        self.squeeze = nn.Conv2d(in_ch, s1x1, 1)            # 1x1 squeeze: reduce channels
        self.sbn = nn.BatchNorm2d(s1x1)
        self.expand1 = nn.Conv2d(s1x1, e1x1, 1)             # 1x1 expand
        self.expand3 = nn.Conv2d(s1x1, e3x3, 3, padding=1)  # 3x3 expand (padding keeps H,W)
        self.ebn = nn.BatchNorm2d(e1x1 + e3x3)
        self.act = nn.ReLU(inplace=True)

    def forward(self, x):
        x = self.act(self.sbn(self.squeeze(x)))             # squeeze -> ReLU
        y1 = self.expand1(x)                                # 1x1 expand branch
        y3 = self.expand3(x)                                # 3x3 expand branch
        merged = torch.cat([y1, y3], 1)                     # concat on channel axis
        return self.act(self.ebn(merged))                   # -> ReLU

### Step 3 — Build a Fire net and a plain-conv baseline

To show the win, we need two networks to compare. `PlainNet` uses only ordinary `3×3` convolutions. `FireNet` stacks four Fire modules around a single late-placed max-pool (downsampling late keeps feature maps large, the paper's Strategy 3).

The `SR` (squeeze ratio) sets how thin the squeeze is relative to the expand width — the knob we sweep in Step 4. Both nets end with global average pooling into a linear classifier.

In [ ]:
# A tiny SqueezeNet-style net (Fire modules) and a plain-conv baseline.
def n_params(m):
    return sum(p.numel() for p in m.parameters())

class PlainNet(nn.Module):          # ordinary 3x3 convs only
    def __init__(self, ch=32, K=8):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, ch, 3, padding=1), nn.BatchNorm2d(ch), nn.ReLU(inplace=True),
            nn.Conv2d(ch, ch, 3, padding=1), nn.BatchNorm2d(ch), nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(ch, ch, 3, padding=1), nn.BatchNorm2d(ch), nn.ReLU(inplace=True),
            nn.Conv2d(ch, ch, 3, padding=1), nn.BatchNorm2d(ch), nn.ReLU(inplace=True))
        self.head = nn.Linear(ch, K)

    def forward(self, x):
        return self.head(self.net(x).mean(dim=(2, 3)))

class FireNet(nn.Module):           # SR = squeeze ratio = squeeze width / expand width
    def __init__(self, SR=0.5, base=32, K=8):
        super().__init__()
        sq = max(1, int(round(SR * base)))     # squeeze width; expand = base/2 + base/2 = base
        self.stem = nn.Sequential(
            nn.Conv2d(3, base, 3, padding=1), nn.BatchNorm2d(base), nn.ReLU(inplace=True))
        self.f1 = Fire(base, sq, base // 2, base // 2)
        self.f2 = Fire(base, sq, base // 2, base // 2)
        self.pool = nn.MaxPool2d(2)            # downsample LATE (Strategy 3)
        self.f3 = Fire(base, sq, base // 2, base // 2)
        self.f4 = Fire(base, sq, base // 2, base // 2)
        self.head = nn.Linear(base, K)

    def forward(self, x):
        x = self.stem(x)
        x = self.f2(self.f1(x))
        x = self.pool(x)
        x = self.f4(self.f3(x))
        return self.head(x.mean(dim=(2, 3)))   # global average pool -> classifier

### Step 4 — Train both nets and sweep the squeeze ratio

Finally we make a toy task — 8 classes of noisy `16×16` images — and train each net with the same optimiser and seed. We compare parameter counts and test accuracy: the Fire net should reach near-equal accuracy with roughly half the parameters.

Then we sweep `SR` from thin to wide. Watch parameters climb with `SR` while accuracy climbs too but with diminishing returns — the trade-off the paper studies in §5.2.

In [ ]:
# Toy task: 8 noisy image classes.
g = torch.Generator().manual_seed(1)
N, Cc, H, W, K = 1000, 3, 16, 16, 8
y = torch.randint(0, K, (N,), generator=g)
proto = torch.randn(K, Cc, H, W, generator=g)
X = proto[y] + 0.9 * torch.randn(N, Cc, H, W, generator=g)
Xtr, ytr, Xte, yte = X[:750], y[:750], X[750:], y[750:]

def train(net, epochs=120, lr=0.05):
    torch.manual_seed(0)
    opt = torch.optim.SGD(net.parameters(), lr=lr, momentum=0.9, weight_decay=1e-4)
    lf = nn.CrossEntropyLoss()
    for _ in range(epochs):
        net.train()
        opt.zero_grad()
        loss = lf(net(Xtr), ytr)
        loss.backward()
        opt.step()
    net.eval()
    with torch.no_grad():
        return (net(Xte).argmax(1) == yte).float().mean().item()

plain = PlainNet()
fire = FireNet(SR=0.5)
acc_plain = train(plain)
acc_fire = train(fire)

print("\n                 params     test acc")
print("plain 3x3 convs  %6d     %.3f" % (n_params(plain), acc_plain))
print("Fire modules     %6d     %.3f" % (n_params(fire), acc_fire))
print("param ratio plain/fire = %.2fx" % (n_params(plain) / n_params(fire)))
# plain 3x3 convs   29160     1.000
# Fire modules      14088     0.996   (~2x fewer params, near-equal accuracy)

# Squeeze-ratio ablation.
print("\nsqueeze-ratio ablation:")
for SR in [0.125, 0.25, 0.5, 0.75]:
    net = FireNet(SR=SR)
    print("  SR=%.3f  params=%5d  acc=%.3f" % (SR, n_params(net), train(net)))
# SR=0.125 params= 4728 acc=0.840   SR=0.250 params= 7848 acc=0.988
# SR=0.500 params=14088 acc=0.996   SR=0.750 params=20328 acc=1.000
# (varies by hardware/seed; our small run, not the paper's reported number.)

## Visualize the data & results

_Can a Fire-module net match a plain-conv net's accuracy with far fewer parameters — and how does the squeeze ratio trade params for accuracy?_

### Step 1 — Re-declare the nets and toy data for a self-contained run

This panel re-imports torch and re-defines the Fire module, both networks, the toy dataset, and the train helper so it can run independently of the cells above. The logic is identical to the reference implementation; names are kept compact here.

In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(0)
g = torch.Generator().manual_seed(1)
N, C, H, W, K = 1000, 3, 16, 16, 8
y = torch.randint(0, K, (N,), generator=g)
proto = torch.randn(K, C, H, W, generator=g)
X = proto[y] + 0.9 * torch.randn(N, C, H, W, generator=g)
Xtr, ytr, Xte, yte = X[:750], y[:750], X[750:], y[750:]

class Fire(nn.Module):                 # squeeze 1x1 -> ReLU; expand (1x1 || 3x3) -> concat -> ReLU
    def __init__(s, cin, s1, e1, e3):
        super().__init__()
        s.sq = nn.Conv2d(cin, s1, 1)
        s.sbn = nn.BatchNorm2d(s1)
        s.e1 = nn.Conv2d(s1, e1, 1)
        s.e3 = nn.Conv2d(s1, e3, 3, padding=1)
        s.ebn = nn.BatchNorm2d(e1 + e3)
        s.a = nn.ReLU(inplace=True)
    def forward(s, x):
        x = s.a(s.sbn(s.sq(x)))
        return s.a(s.ebn(torch.cat([s.e1(x), s.e3(x)], 1)))

def nparams(m):
    return sum(p.numel() for p in m.parameters())

class PlainNet(nn.Module):
    def __init__(s, ch=32):
        super().__init__()
        s.net = nn.Sequential(
            nn.Conv2d(3, ch, 3, padding=1), nn.BatchNorm2d(ch), nn.ReLU(inplace=True),
            nn.Conv2d(ch, ch, 3, padding=1), nn.BatchNorm2d(ch), nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(ch, ch, 3, padding=1), nn.BatchNorm2d(ch), nn.ReLU(inplace=True),
            nn.Conv2d(ch, ch, 3, padding=1), nn.BatchNorm2d(ch), nn.ReLU(inplace=True))
        s.head = nn.Linear(ch, K)
    def forward(s, x):
        return s.head(s.net(x).mean(dim=(2, 3)))

class FireNet(nn.Module):
    def __init__(s, SR=0.5, base=32):
        super().__init__()
        sq = max(1, int(round(SR * base)))
        s.stem = nn.Sequential(nn.Conv2d(3, base, 3, padding=1), nn.BatchNorm2d(base), nn.ReLU(inplace=True))
        s.f1 = Fire(base, sq, base // 2, base // 2)
        s.f2 = Fire(base, sq, base // 2, base // 2)
        s.pool = nn.MaxPool2d(2)
        s.f3 = Fire(base, sq, base // 2, base // 2)
        s.f4 = Fire(base, sq, base // 2, base // 2)
        s.head = nn.Linear(base, K)
    def forward(s, x):
        x = s.stem(x)
        x = s.f2(s.f1(x))
        x = s.pool(x)
        x = s.f4(s.f3(x))
        return s.head(x.mean(dim=(2, 3)))

def train(net, epochs=120, lr=0.05):
    torch.manual_seed(0)
    opt = torch.optim.SGD(net.parameters(), lr=lr, momentum=0.9, weight_decay=1e-4)
    lf = nn.CrossEntropyLoss()
    for _ in range(epochs):
        net.train()
        opt.zero_grad()
        lf(net(Xtr), ytr).backward()
        opt.step()
    net.eval()
    with torch.no_grad():
        return (net(Xte).argmax(1) == yte).float().mean().item()

### Step 2 — Compare params vs accuracy and sweep the squeeze ratio

Now we train the plain net and the Fire net and print their parameter counts and test accuracy side by side, then sweep `SR` across four values. The numbers behind the panel: the Fire net matches the plain net at roughly half the parameters, and accuracy rises with `SR` at a growing parameter cost.

In [ ]:
plain = PlainNet()
fire = FireNet(SR=0.5)
print("plain params=%d acc=%.3f" % (nparams(plain), train(plain)))
print("fire  params=%d acc=%.3f" % (nparams(fire), train(fire)))

for SR in [0.125, 0.25, 0.5, 0.75]:
    net = FireNet(SR=SR)
    print("SR=%.3f params=%d acc=%.3f" % (SR, nparams(net), train(net)))
# plain params=29160 acc=1.000 ; fire params=14088 acc=0.996 (~2x fewer, near-equal)
# SR=0.125 p=4728 a=0.840 | SR=0.25 p=7848 a=0.988 | SR=0.5 p=14088 a=0.996 | SR=0.75 p=20328 a=1.000
# Our small run, not the paper's reported number.

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** The squeeze-ratio ablation. You have a working tiny SqueezeNet-style net. Rebuild it at several
            squeeze ratios &mdash; making the squeeze layer thinner (small SR) or wider (large SR) &mdash; while
            keeping everything else identical, and retrain each. What happens to the parameter count and the test
            accuracy as SR changes, and what does the trend demonstrate?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Change one thing only: the squeeze width $s_{1\times1}$ (via SR = squeeze / expand width). Keep depth, expand widths, optimizer, data, and seed identical. — _An honest ablation isolates the squeeze ratio; any change in accuracy or params is attributable to it alone._
- Record the parameter count and test accuracy at each SR. — _A thinner squeeze multiplies both expand terms by a smaller number, so it directly shrinks the parameter count &mdash; the lever from the param formula._
- Plot accuracy vs parameters across SR values. — _It reveals the trade-off: more squeeze width buys accuracy at a parameter cost, with diminishing returns &mdash; the &sect;5.2 finding._

**Answer:** Parameter count rises monotonically with SR (a wider squeeze = more weights), and accuracy
                 rises with it too, but with diminishing returns. In our small CPU run a thin squeeze
                 (SR=0.125, 4728 params) reached 0.840 test accuracy, SR=0.25 (7848 params) reached 0.988,
                 SR=0.5 (14088 params) reached 0.996, and SR=0.75 (20328 params) reached 1.000. So you can trade
                 a few points of accuracy for a big parameter cut by squeezing harder &mdash; exactly the
                 SR trade-off the paper explores in &sect;5.2. The CODEVIZ panel plots this curve. (Our small run,
                 not the paper's reported numbers.)

</details>

**Problem 2.** A Fire module has $C=64$ input channels, squeeze $s_{1\times1}=16$, and expand
            $e_{1\times1}=64$, $e_{3\times3}=64$. Compute its parameter count (weights + biases), and compare to
            one ordinary $3\times3$ convolution doing $64\to128$ on the same input.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Squeeze $1\times1$ ($64\to16$): $64\cdot16 = 1024$ weights $+\,16$ biases $=1040$. — _Standard conv rule $a\cdot b\cdot k^2 + b$ with $k=1$._
- Expand $1\times1$ ($16\to64$): $16\cdot64 = 1024$ weights $+\,64$ biases $=1088$. Expand $3\times3$ ($16\to64$): $9\cdot16\cdot64 = 9216$ weights $+\,64$ biases $=9280$. — _The $3\times3$ reads only the 16 squeezed channels, not the full 64 — Strategy 2._
- Standard conv ($64\to128$, $3\times3$): $9\cdot64\cdot128 = 73728$ weights $+\,128$ biases $=73856$. — _One full $3\times3$ conv reading all 64 input channels and writing 128._

**Answer:** The Fire module totals $1040 + 1088 + 9280 = \mathbf{11{,}408}$ parameters and outputs
                 $64+64=128$ channels. A single standard $3\times3$ convolution producing the same 128 output
                 channels costs $\mathbf{73{,}856}$ parameters &mdash; about $6.5\times$ more. The Fire module is
                 dramatically cheaper because the expensive $3\times3$ filters read only the 16 squeezed channels
                 instead of the full 64 (Strategy 2), and because the channel-mixing is done by cheap
                 $1\times1$ filters (Strategy 1).

</details>

**Problem 3.** Why does the squeeze layer come before the $3\times3$ filters in a Fire module, rather than
            applying the $3\times3$ filters straight to the module's full input? Tie your answer to the design
            strategies.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Recall a conv layer's weight count grows with its INPUT-channel count. — _Weights $=(\text{in})\times(\text{out})\times(\text{filter area})$; halving the inputs halves the weights._
- Note the $3\times3$ filter has $9\times$ the area of a $1\times1$, so it is the costly one to feed. — _Strategy 1: $3\times3$ filters are 9× more expensive per (in,out) pair than $1\times1$._
- Put a thin $1\times1$ squeeze first so the $3\times3$ sees only $s_{1\times1}$ inputs, not the full $C$. — _Strategy 2: shrink the input-channel count to the $3\times3$ filters before they run._

**Answer:** Because the $3\times3$ convolution is the expensive one (it has $9\times$ the filter area of a
                 $1\times1$, Strategy 1), and a convolution's weight count is proportional to its
                 input-channel count. The squeeze layer is a cheap $1\times1$ that first drops the channel
                 count from the full $C$ down to a small $s_{1\times1}$, so when the $3\times3$ filters run they
                 read only $s_{1\times1}$ channels instead of $C$ (Strategy 2). Applying the $3\times3$
                 filters straight to the full input would multiply their weight count by $C/s_{1\times1}$ &mdash;
                 in the worked example, $32/8 = 4\times$ more &mdash; defeating the savings. Squeeze-then-expand is
                 how the module obeys both Strategy 1 and Strategy 2 at once.

</details>